In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline

import sys
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = './regressao_Q2.csv'
df = pd.read_csv(data)

In [5]:
df.shape

(2500, 13)

In [6]:
df.head()

,v_1,v_2,v_3,v_4,v_5,v_6,v_7,v_8,v_9,v_10,v_11,v_12,target
0,1.92864,1.48414,0.86814,-0.67666,-0.28747,-1.45108,-0.73662,0.03134,-0.53872,1.30562,0.11557,-0.30478,127.682465
1,0.22185,-0.55320,-0.29845,0.65870,-0.30132,1.49319,-0.43096,0.33835,-0.30827,1.25765,1.88584,-0.57726,50.022972
2,-0.02183,0.13602,-0.37426,-1.29096,0.71912,1.95088,1.99309,-1.24197,-2.15377,-2.01455,-0.84625,0.29845,-24.364369
3,0.86528,1.36937,1.27999,1.18124,-0.72465,-0.02175,0.40340,-0.28272,-0.44390,0.84051,0.03326,-0.98550,96.415408
4,1.41333,-0.02920,-0.67228,1.76116,-1.12178,0.18002,0.48476,-0.76394,-0.16421,-2.42048,0.79727,-0.44033,157.407129


In [ ]:
X = df.drop(['target'], axis=1)
y = df['target']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state = 42)

X_train.shape, X_test.shape

((1675, 12), (825, 12))

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Definir o modelo SVM de Regressão com kernel linear e fator de regularização C=0.01
# Nota: C é o inverso do parâmetro de regularização (menor C = maior regularização)
svr_model = SVR(kernel='linear', C=0.01)

# Validação cruzada sequencial (Time Series Split) com 5 folds
# Esta técnica é apropriada para dados temporais
tss = TimeSeriesSplit(n_splits=5)

# Realizar validação cruzada sequencial com diferentes métricas
print("=== Validação Cruzada Sequencial com SVR ===")

# R² Score (coeficiente de determinação)
r2_scores = cross_val_score(svr_model, X, y, cv=tss, scoring='r2')
print("R² Score em cada fold:")
for i, score in enumerate(r2_scores, 1):
    print(f"Fold {i}: {score:.4f}")
print(f"R² médio: {r2_scores.mean():.4f}")
print(f"Desvio padrão R²: {r2_scores.std():.4f}")

# Mean Squared Error (negativo, por isso multiplicamos por -1)
mse_scores = cross_val_score(svr_model, X, y, cv=tss, scoring='neg_mean_squared_error')
mse_scores = -mse_scores  # Converter para valores positivos
print(f"\nMSE médio: {mse_scores.mean():.4f}")
print(f"Desvio padrão MSE: {mse_scores.std():.4f}")

# Mean Absolute Error (negativo, por isso multiplicamos por -1)
mae_scores = cross_val_score(svr_model, X, y, cv=tss, scoring='neg_mean_absolute_error')
mae_scores = -mae_scores  # Converter para valores positivos
print(f"\nMAE médio: {mae_scores.mean():.4f}")
print(f"Desvio padrão MAE: {mae_scores.std():.4f}")

# Treinar o modelo final com todos os dados
svr_model.fit(X, y)

print("\n=== Modelo SVR treinado com sucesso! ===")
print(f"Número de vetores de suporte: {svr_model.n_support_}")
print(f"Parâmetros do modelo:")
print(f"  - Kernel: {svr_model.kernel}")
print(f"  - C (regularização): {svr_model.C}")
print(f"  - Epsilon: {svr_model.epsilon}")

=== Validação Cruzada Sequencial com SVR ===
R² Score em cada fold:
Fold 1: 0.0394
Fold 2: 0.0866
Fold 3: 0.1241
Fold 4: 0.1705
Fold 5: 0.2090
R² médio: 0.1259
Desvio padrão R²: 0.0599

MSE médio: 18153.6292
Desvio padrão MSE: 994.0656
R² Score em cada fold:
Fold 1: 0.0394
Fold 2: 0.0866
Fold 3: 0.1241
Fold 4: 0.1705
Fold 5: 0.2090
R² médio: 0.1259
Desvio padrão R²: 0.0599

MSE médio: 18153.6292
Desvio padrão MSE: 994.0656

MAE médio: 107.4224
Desvio padrão MAE: 2.5278

=== Modelo SVR treinado com sucesso! ===
Número de vetores de suporte: [2496]
Parâmetros do modelo:
  - Kernel: linear
  - C (regularização): 0.01
  - Epsilon: 0.1

MAE médio: 107.4224
Desvio padrão MAE: 2.5278

=== Modelo SVR treinado com sucesso! ===
Número de vetores de suporte: [2496]
Parâmetros do modelo:
  - Kernel: linear
  - C (regularização): 0.01
  - Epsilon: 0.1


In [13]:
# Calculando a média da métrica MSE no treino e no teste usando validação cruzada
from sklearn.model_selection import validation_curve
import numpy as np

# Usando validation_curve para calcular MSE no treino e teste
# Vamos usar o mesmo parâmetro C=0.01 que definimos anteriormente
train_scores, test_scores = validation_curve(
    SVR(kernel='linear'), 
    X, y, 
    param_name='C', 
    param_range=[0.01],  # Usando o mesmo C=0.01
    cv=TimeSeriesSplit(n_splits=5), 
    scoring='neg_mean_squared_error'
)

# Converter valores negativos para positivos (MSE)
train_mse = -train_scores
test_mse = -test_scores

# Calcular e exibir as médias
print("=== Média da Métrica MSE ===")
print(f"MSE médio no treino: {train_mse.mean():.4f}")
print(f"Desvio padrão MSE treino: {train_mse.std():.4f}")
print(f"MSE médio no teste: {test_mse.mean():.4f}")
print(f"Desvio padrão MSE teste: {test_mse.std():.4f}")

# Exibir MSE por fold
print("\n=== MSE por Fold ===")
for i in range(len(train_mse[0])):
    print(f"Fold {i+1}:")
    print(f"  Treino: {train_mse[0][i]:.4f}")
    print(f"  Teste:  {test_mse[0][i]:.4f}")

# Diferença entre treino e teste (indicador de overfitting)
diff_mse = train_mse.mean() - test_mse.mean()
print(f"\nDiferença MSE (treino - teste): {diff_mse:.4f}")
if diff_mse > 0:
    print("↳ Modelo pode estar com overfitting (MSE treino > MSE teste)")
elif diff_mse < 0:
    print("↳ Modelo pode estar com underfitting (MSE treino < MSE teste)")
else:
    print("↳ Modelo bem balanceado")

=== Média da Métrica MSE ===
MSE médio no treino: 17400.5067
Desvio padrão MSE treino: 996.7715
MSE médio no teste: 18153.6292
Desvio padrão MSE teste: 994.0656

=== MSE por Fold ===
Fold 1:
  Treino: 18784.5272
  Teste:  19414.3180
Fold 2:
  Treino: 18208.8207
  Teste:  17660.8112
Fold 3:
  Treino: 17214.0324
  Teste:  19110.5038
Fold 4:
  Treino: 16817.3221
  Teste:  16697.2683
Fold 5:
  Treino: 15977.8311
  Teste:  17885.2446

Diferença MSE (treino - teste): -753.1225
↳ Modelo pode estar com underfitting (MSE treino < MSE teste)
